# Sauti — Démo intégrée (bout en bout)

La boucle complète, avec les **vrais modèles Kiriku** et **ton propre code** (cloné depuis GitHub) :

> 🎙️ **audio wolof** → 📝 **transcription (ASR)** → 🚨 **détection de danger** → 🔊 **réponse vocale (TTS)**

**Avant de commencer :** pousse tes derniers fichiers sur GitHub (base 30 questions,
`signes_danger.json` avec mots-clés wolof, `tts.py` avec `.lower()`), pour que le clone
soit à jour :
```bash
git add . && git commit -m "MAJ contenu + danger wolof" && git push
```

⚠️ GPU recommandé · après l'installation, **redémarre la session** si Colab le propose.


## 1. Installation (ASR + TTS dans une seule session)

In [ ]:
!pip -q install coqui-tts "transformers<5" librosa soundfile huggingface_hub jiwer
# Si Colab affiche "RESTART SESSION", clique dessus AVANT de continuer.

## 2. Connexion Hugging Face

In [ ]:
from huggingface_hub import login
from getpass import getpass
login(getpass("Token HF (read) : "))

## 3. Cloner ton dépôt et charger les modèles

In [ ]:
import os, sys, torch

# Cloner le repo (une fois)
if not os.path.exists("sauti"):
    !git clone -q https://github.com/compo90/sauti
sys.path.insert(0, "sauti/src"); sys.path.insert(0, "sauti")

# --- ASR (M-Kiriku multilingue) ---
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="AIHubSN/M-Kiriku-ASR",
               device=0 if torch.cuda.is_available() else -1)

# --- TTS (Kiriku Wolof, Coqui VITS) ---
from huggingface_hub import snapshot_download
from TTS.utils.synthesizer import Synthesizer
snapshot_download(repo_id="AIHubSN/Kiriku-Wolof-TTS", local_dir="tts_wolof")
tts = Synthesizer(tts_checkpoint="tts_wolof/model.pth",
                  tts_config_path="tts_wolof/config.json",
                  use_cuda=torch.cuda.is_available())

# --- Détecteur de danger (TON code + mots-clés wolof) ---
from sauti.nlu.danger_detector import DangerDetector
danger = DangerDetector()
print("Tout est chargé ✅")

## 4. Réponses vocales de la démo (en wolof)
Réponses courtes validées par un locuteur natif. À terme, elles viennent de la base
de connaissances validée cliniquement.

In [ ]:
# Message d'urgence (déclenché si un signe de danger est détecté)
REPONSE_URGENCE = ("lii amul xaar : demal léegi ci barabu fajjukaay bi la gën a jege, "
                   "walla poste de santé. àndal ak keneen.")

# Message par défaut (question sans danger) — à enrichir depuis la base
REPONSE_INFO = "jërëjëf ci sa laaj. ngir wérgu-yaram, demal ci poste de santé bi."


## 5. La boucle complète
Uploade un court audio wolof (une question ou un signe de danger) et écoute la réponse.

In [ ]:
from google.colab import files
from IPython.display import Audio, display

up = files.upload()
audio_path = list(up.keys())[0]

# 1) ASR : écoute
texte = asr(audio_path)["text"].strip()
print("🎙️  Transcription :", texte)

# 2) Triage danger (ton DangerDetector + mots-clés wolof)
d = danger.analyser(texte, langue="wol")

# 3) Choix de la réponse
if d.est_danger:
    print(f"🚨 DANGER détecté : {d.signes}  (action : {d.action})")
    reponse = REPONSE_URGENCE
else:
    print("✅ Pas de signe de danger")
    reponse = REPONSE_INFO
print("💬 Réponse :", reponse)

# 4) TTS : réponse vocale (minuscules = critère Kiriku)
wav = tts.tts(reponse.lower())
tts.save_wav(wav, "reponse.wav")
print("🔊 Réponse vocale :")
display(Audio("reponse.wav"))

---
## C'est ça, Sauti.
Tu viens de dérouler la boucle complète avec les vrais modèles Kiriku et ton propre code :
une femme parle en wolof, le système comprend, détecte l'urgence, et répond par la voix.

**Pour la vidéo démo :** lance la cellule 5 deux fois —
1. une question **bénigne** (« que dois-je manger ? ») → réponse info,
2. un **signe de danger** (« amna dëret » / « sama yaram tàng na lool ») → alerte + orientation.

Le contraste entre les deux, c'est ce qui prouve la valeur du projet.
